# Notebook 1A — Set Up SageMaker MLflow App for ITI113

This notebook creates or reuses a **SageMaker MLflow App** for experiment tracking, then performs a small test MLflow run.

The notebook is now team-parameterised. Update only the configuration cell:

- `TEAM_ID`
- `STUDENT_ID`
- `PROJECT_NAME`

The MLflow App is created with tags such as `Course`, `TeamId`, and `ProjectName`. These tags support IAM team-level restriction, for example allowing Team 40 to use only MLflow Apps tagged `TeamId=team40`.

This is an AWS-native alternative to the previous Databricks MLflow setup. The SageMaker training pipeline itself does **not** need to change; only the MLflow tracking target changes.

> Important: For strict team isolation, the team role IAM policy should allow/deny MLflow access based on the MLflow App `TeamId` tag.


## 0. What this notebook does

1. Checks whether the installed AWS SDK supports SageMaker MLflow App APIs.
2. Sets the course/team/student/project configuration.
3. Creates or reuses a SageMaker MLflow App.
4. Creates the MLflow App with these tags when it does not already exist:
   - `Course`
   - `Semester`
   - `TeamId`
   - `StudentId`
   - `ProjectName`
   - `CreatedByNotebook`
5. If the App already exists, checks its tags and tries to add/update the required tags.
6. Prints the MLflow App ARN, which becomes the MLflow tracking URI.
7. Creates a presigned MLflow UI URL.
8. Runs a small MLflow test run using the configured team/student metadata.

Expected result:

```text
MLflow tracking URI = arn:aws:sagemaker:ap-southeast-1:<account-id>:mlflow-app/...
Experiment          = ITI113/<team-id>/Experiment1
Run name            = <team-id>_<student-id>_mlflow_app_test
```


## 1. Install or update required packages

The `sagemaker-mlflow` plugin lets the normal MLflow Python client authenticate to SageMaker MLflow using AWS IAM/SigV4. Restart the kernel after this cell if the notebook asks you to.

In [1]:
%pip install -U boto3 botocore mlflow sagemaker-mlflow

  Using cached mlflow-3.15.1-py3-none-any.whl.metadata (49 kB)


  Using cached mlflow_skinny-3.15.1-py3-none-any.whl.metadata (50 kB)


  Using cached mlflow_tracing-3.15.1-py3-none-any.whl.metadata (19 kB)


  Using cached prettytable-3.18.0-py3-none-any.whl.metadata (37 kB)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/15.6 MB ? eta -:--:--

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [mlflow]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [mlflow]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [mlflow]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [mlflow]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
aiobotocore 3.8.0 requires botocore<1.43.47,>=1.43.3, but you have botocore 1.43.72 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 21.0.0 which is incompatible.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.6.0 which is incompatible.
sagemaker-studio-analytics-extension 0.3.0 requires sparkmagic==0.22.0, but you have sparkmagic 0.21.0 whi

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Update only these values if your bucket, role, or team naming is different.

The role used by the MLflow App must be able to access the S3 artifact store. For the current Team01 classroom setup, the expected role is:

```text
arn:aws:iam::<account-id>:role/SageMakerExecutionRole-ITI113-Team01
```


In [2]:
import boto3
from botocore.exceptions import ClientError
import time
import json
from datetime import datetime
from pathlib import Path

REGION = "ap-southeast-1"
COURSE = "ITI113"
SEMESTER = "26S1"

# Change these for each team/student.
TEAM_ID = "team05"
STUDENT_ID = "s502"

# Project name used in tags and artifact organisation.
PROJECT_NAME = "Heart_Attack_Risk_Assessment"

# Existing course bucket from the SageMaker Pipeline lab.
CLASS_BUCKET = "nyp-26s1-iti113"

# One MLflow App per team is usually enough.
MLFLOW_APP_NAME = f"iti113-26s1-{TEAM_ID}-mlflow-app"

# Where MLflow run artifacts will be stored.
ARTIFACT_STORE_URI = f"s3://{CLASS_BUCKET}/iti113/{TEAM_ID}/mlflow-app-artifacts/"

# Experiment name inside MLflow App. Use a normal MLflow experiment name, not a Databricks /Workspace path.
EXPERIMENT_NAME = f"{COURSE}/{TEAM_ID}/Experiment1"

# Optional: make this MLflow App the account/domain default. Keep False for classroom safety.
SET_AS_ACCOUNT_DEFAULT = False
SET_AS_DEFAULT_FOR_EXISTING_DOMAINS = False

session = boto3.Session(region_name=REGION)
sts = session.client("sts")
sm = session.client("sagemaker")
s3 = session.client("s3")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
CALLER_ARN = sts.get_caller_identity()["Arn"]

# Build role name from TEAM_ID, e.g. team40 -> SageMakerExecutionRole-ITI113-Team40.
TEAM_ROLE_SUFFIX = TEAM_ID.lower().replace("team", "Team")
ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/SageMakerExecutionRole-ITI113-{TEAM_ROLE_SUFFIX}"

# Required tags for team-level MLflow IAM restriction.
MLFLOW_APP_TAGS = [
    {"Key": "Course", "Value": COURSE},
    {"Key": "Semester", "Value": SEMESTER},
    {"Key": "TeamId", "Value": TEAM_ID},
    {"Key": "StudentId", "Value": STUDENT_ID},
    {"Key": "ProjectName", "Value": PROJECT_NAME},
    {"Key": "CreatedByNotebook", "Value": "01A_setup_sagemaker_mlflow_app"},
]

print("Account:", ACCOUNT_ID)
print("Caller ARN:", CALLER_ARN)
print("Region:", REGION)
print("Team ID:", TEAM_ID)
print("Student ID:", STUDENT_ID)
print("Project Name:", PROJECT_NAME)
print("MLflow App Name:", MLFLOW_APP_NAME)
print("Artifact Store:", ARTIFACT_STORE_URI)
print("Role ARN:", ROLE_ARN)
print("Experiment:", EXPERIMENT_NAME)
print("\nMLflow App tags to apply:")
for tag in MLFLOW_APP_TAGS:
    print(f"  {tag['Key']} = {tag['Value']}")


Account: 044528205969
Caller ARN: arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team05/SageMaker
Region: ap-southeast-1
Team ID: team05
Student ID: s502
Project Name: Heart_Attack_Risk_Assessment
MLflow App Name: iti113-26s1-team05-mlflow-app
Artifact Store: s3://nyp-26s1-iti113/iti113/team05/mlflow-app-artifacts/
Role ARN: arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team05
Experiment: ITI113/team05/Experiment1

MLflow App tags to apply:
  Course = ITI113
  Semester = 26S1
  TeamId = team05
  StudentId = s502
  ProjectName = Heart_Attack_Risk_Assessment
  CreatedByNotebook = 01A_setup_sagemaker_mlflow_app


### Verify the SageMaker Studio project folder

Run the next cell before creating the MLflow App. The notebook should be opened from your `Heart_Attack_Risk_Assessment` project folder. The generated configuration JSON will be saved in this same working directory.


In [3]:
from pathlib import Path

PROJECT_FOLDER_NAME = "Heart_Attack_Risk_Assessment"
CURRENT_WORKING_DIR = Path.cwd().resolve()

print("Current working directory:")
print(CURRENT_WORKING_DIR)

print("\nExpected project folder name:")
print(PROJECT_FOLDER_NAME)

if CURRENT_WORKING_DIR.name != PROJECT_FOLDER_NAME:
    print("\n[WARNING] The current working directory does not end with the expected project folder name.")
    print("You can still continue, but the generated JSON will be saved in the directory shown above.")
else:
    print("\n[OK] Notebook is running from the expected project folder.")


Current working directory:
/home/sagemaker-user/Heart_Attack_Risk_Assessment

Expected project folder name:
Heart_Attack_Risk_Assessment

[OK] Notebook is running from the expected project folder.


## 3. Check SDK/API support

If this cell says `create_mlflow_app` is missing, update the Studio environment's `boto3` and `botocore`, then restart the kernel and rerun the notebook.


In [4]:
required_methods = [
    "create_mlflow_app",
    "list_mlflow_apps",
    "describe_mlflow_app",
    "create_presigned_mlflow_app_url",
    "add_tags",
    "list_tags",
]

missing = [method for method in required_methods if not hasattr(sm, method)]

print("SageMaker client supports:")
for method in required_methods:
    print(f"  {method}: {hasattr(sm, method)}")

if missing:
    raise RuntimeError(
        "Your boto3/botocore version does not support these SageMaker MLflow App/tag APIs: "
        + ", ".join(missing)
        + "\nRun the package update cell, restart the kernel, and rerun."
    )


SageMaker client supports:
  create_mlflow_app: True
  list_mlflow_apps: True
  describe_mlflow_app: True
  create_presigned_mlflow_app_url: True
  add_tags: True
  list_tags: True


## 4. Verify the S3 artifact store prefix is writable

This checks that the current role can write a small test file to the MLflow artifact prefix.


In [5]:
from urllib.parse import urlparse

def parse_s3_uri(uri: str):
    parsed = urlparse(uri)
    if parsed.scheme != "s3":
        raise ValueError(f"Not an S3 URI: {uri}")
    return parsed.netloc, parsed.path.lstrip("/")

artifact_bucket, artifact_prefix = parse_s3_uri(ARTIFACT_STORE_URI)
test_key = f"{artifact_prefix.rstrip('/')}/_setup_test/{TEAM_ID}_{STUDENT_ID}_write_test.txt"

try:
    s3.put_object(
        Bucket=artifact_bucket,
        Key=test_key,
        Body=(
            f"MLflow App setup write test at {datetime.utcnow().isoformat()}Z\n"
        ).encode("utf-8"),
    )
    print("S3 write test succeeded:")
    print(f"s3://{artifact_bucket}/{test_key}")
except ClientError:
    print("S3 write test failed. Check bucket/prefix permissions.")
    raise


S3 write test succeeded:
s3://nyp-26s1-iti113/iti113/team05/mlflow-app-artifacts/_setup_test/team05_s502_write_test.txt


/tmp/ipykernel_946/1603707423.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"MLflow App setup write test at {datetime.utcnow().isoformat()}Z\n"


## 5. Optionally discover SageMaker Studio domain IDs

This is only needed if you want the MLflow App to be configured as the default for one or more Studio domains. For the classroom, it is safer to leave `SET_AS_DEFAULT_FOR_EXISTING_DOMAINS = False`.


In [6]:
domain_ids = []

try:
    domains_response = sm.list_domains()
    domain_ids = [domain["DomainId"] for domain in domains_response.get("Domains", [])]
    print("Studio domains found:", domain_ids)
except ClientError as e:
    print("Could not list Studio domains. This is okay if you are not setting defaults.")
    print(e)


Studio domains found: ['d-popmr5pqbh1n', 'd-wcuptup1pj6w', 'd-gpdrdk2w4dgw', 'd-5q0cdlsfisve', 'd-lsrccj8tewjf', 'd-oijl6tgx8os2', 'd-cydcdxkc4yot', 'd-tjsofl2bch3a', 'd-8nb3rzhhmygx', 'd-jgu4uwvdu9ir']


## 6. Create or reuse the SageMaker MLflow App

This cell first checks whether an MLflow App with the configured name already exists.

- If the App does **not** exist, it creates the App with the required tags.
- If the App already exists, it reuses the App and checks whether the required tags are present.

These tags are important for team-level restriction. For example, Team 40's role can be restricted to use only MLflow Apps where:

```text
aws:ResourceTag/TeamId = team40
```

Creation may take a few minutes. The notebook polls until the App status becomes `Created` or `Updated`.


In [7]:
def find_mlflow_app_by_name(name: str):
    paginator = sm.get_paginator("list_mlflow_apps")
    for page in paginator.paginate():
        for summary in page.get("Summaries", []):
            if summary.get("Name") == name:
                return summary
    return None


def ensure_mlflow_app_tags(resource_arn: str, required_tags: list):
    """Check and apply required tags to an existing MLflow App.

    If the current role does not have sagemaker:AddTags/ListTags permission,
    this function will print a warning and continue. The admin can tag the App later.
    """
    required = {tag["Key"]: tag["Value"] for tag in required_tags}

    try:
        existing_tags_response = sm.list_tags(ResourceArn=resource_arn)
        existing = {
            tag["Key"]: tag["Value"]
            for tag in existing_tags_response.get("Tags", [])
        }

        missing_or_different = [
            {"Key": key, "Value": value}
            for key, value in required.items()
            if existing.get(key) != value
        ]

        if missing_or_different:
            print("\nAdding/updating required MLflow App tags:")
            for tag in missing_or_different:
                print(f"  {tag['Key']} = {tag['Value']}")

            sm.add_tags(
                ResourceArn=resource_arn,
                Tags=missing_or_different,
            )
        else:
            print("\nExisting MLflow App already has the required tags.")

        final_tags = sm.list_tags(ResourceArn=resource_arn).get("Tags", [])
        print("\nCurrent MLflow App tags:")
        for tag in final_tags:
            print(f"  {tag['Key']} = {tag['Value']}")

    except ClientError as e:
        print("\n[WARNING] Could not verify or update MLflow App tags.")
        print("This may happen if the current role does not have sagemaker:ListTags/AddTags.")
        print("Ask the admin to ensure these tags exist on the MLflow App:")
        for tag in required_tags:
            print(f"  {tag['Key']} = {tag['Value']}")
        print("\nOriginal error:")
        print(e)


existing = find_mlflow_app_by_name(MLFLOW_APP_NAME)

if existing:
    mlflow_app_arn = existing["Arn"]
    print("Reusing existing MLflow App:")
    print(json.dumps(existing, indent=2, default=str))

    # Important for team-level MLflow IAM restriction.
    ensure_mlflow_app_tags(mlflow_app_arn, MLFLOW_APP_TAGS)

else:
    create_args = {
        "Name": MLFLOW_APP_NAME,
        "ArtifactStoreUri": ARTIFACT_STORE_URI,
        "RoleArn": ROLE_ARN,
        "ModelRegistrationMode": "AutoModelRegistrationDisabled",
        "Tags": MLFLOW_APP_TAGS,
    }

    if SET_AS_ACCOUNT_DEFAULT:
        create_args["AccountDefaultStatus"] = "ENABLED"

    if SET_AS_DEFAULT_FOR_EXISTING_DOMAINS and domain_ids:
        create_args["DefaultDomainIdList"] = domain_ids

    print("Creating MLflow App with args:")
    print(json.dumps(create_args, indent=2, default=str))

    response = sm.create_mlflow_app(**create_args)
    mlflow_app_arn = response["Arn"]
    print("Create response:", response)

print("\nMLflow App ARN:")
print(mlflow_app_arn)


Reusing existing MLflow App:
{
  "Arn": "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-ZOP45XZVBR5D",
  "Name": "iti113-26s1-team05-mlflow-app",
  "Status": "Created",
  "CreationTime": "2026-07-23 11:38:47+00:00",
  "LastModifiedTime": "2026-08-01 16:09:19.917000+00:00",
  "MlflowVersion": "3.10.1"
}

Existing MLflow App already has the required tags.

Current MLflow App tags:
  sagemaker:user-profile-arn = arn:aws:sagemaker:ap-southeast-1:044528205969:user-profile/d-zjad5kjaiedi/team05-s501
  Semester = 26S1
  sagemaker:domain-arn = arn:aws:sagemaker:ap-southeast-1:044528205969:domain/d-zjad5kjaiedi
  ProjectName = Heart_Attack_Risk_Assessment
  sagemaker:space-arn = arn:aws:sagemaker:ap-southeast-1:044528205969:space/d-zjad5kjaiedi/project-space-team05
  Course = ITI113
  TeamId = team05
  CreatedByNotebook = 01A_setup_sagemaker_mlflow_app
  StudentId = s502

MLflow App ARN:
arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-ZOP45XZVBR5D


In [8]:
def wait_for_mlflow_app(arn: str, timeout_seconds: int = 600, poll_seconds: int = 20):
    start = time.time()
    last_status = None

    while True:
        desc = sm.describe_mlflow_app(Arn=arn)
        status = desc.get("Status")

        if status != last_status:
            print(f"Status: {status}")
            last_status = status

        if status in ["Created", "Updated"]:
            return desc

        if status in ["CreateFailed", "UpdateFailed", "DeleteFailed", "Deleted"]:
            raise RuntimeError(
                f"MLflow App entered failure status: {status}\n"
                + json.dumps(desc, indent=2, default=str)
            )

        if time.time() - start > timeout_seconds:
            raise TimeoutError(f"Timed out waiting for MLflow App. Last status: {status}")

        time.sleep(poll_seconds)

mlflow_app_desc = wait_for_mlflow_app(mlflow_app_arn)

print("\nFinal MLflow App description:")
print(json.dumps(mlflow_app_desc, indent=2, default=str))


Status: Created

Final MLflow App description:
{
  "Arn": "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-ZOP45XZVBR5D",
  "Name": "iti113-26s1-team05-mlflow-app",
  "ArtifactStoreUri": "s3://nyp-26s1-iti113/iti113/team05/mlflow-app-artifacts/",
  "MlflowVersion": "3.10.1",
  "RoleArn": "arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team05",
  "Status": "Created",
  "ModelRegistrationMode": "AutoModelRegistrationDisabled",
  "CreationTime": "2026-07-23 11:38:47+00:00",
  "CreatedBy": {
    "UserProfileArn": "arn:aws:sagemaker:ap-southeast-1:044528205969:user-profile/d-zjad5kjaiedi/team05-s501",
    "UserProfileName": "team05-s501",
    "DomainId": "d-zjad5kjaiedi",
    "IamIdentity": {
      "Arn": "arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team05/SageMaker"
    }
  },
  "LastModifiedTime": "2026-08-01 16:09:19.917000+00:00",
  "LastModifiedBy": {
    "UserProfileArn": "arn:aws:sagemaker:ap-southeast-1:044528205969:user-profile/d-

## 7. Get a presigned MLflow UI URL

The URL is usually single-use and expires. Generate a fresh URL whenever you want to open the MLflow UI.


In [9]:
url_response = sm.create_presigned_mlflow_app_url(
    Arn=mlflow_app_arn,
    ExpiresInSeconds=300,
    SessionExpirationDurationInSeconds=3600,
)

mlflow_ui_url = url_response["AuthorizedUrl"]

print("Open this MLflow UI URL in a browser tab:")
print(mlflow_ui_url)


Open this MLflow UI URL in a browser tab:
https://app-ZOP45XZVBR5D.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkNWTDVBUiIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNHpONkx3MFdDbjRGMEpadDEvemU5cURmWEJ1SXQvR3cwUm13UlRLTExmYVlBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGcU0ySlhZV2RUVkdwT1RrTkRRbkJyVkhSYVpUaGFlbTFaTW5RM1VUSkdlbVpxZEc1WlUxWk5lSGw0YlZwSlZHd3hlalZsYlc4elIxQklWbHBST1V0UlFUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFUenR4cnZ3RjhaYWRQK1VCei9NN2pzQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF3VXZUTVlyVmllM0pZNm93SUNBUkNBTzJXdUtqWWFPZUMzZ2xaOGFGMWVrUnNydE9UU0UraHRMdUxSbkV1WnNyc3kzRVViL2VGYk1WU25qL0tYckp6UHZMM2VKZmtab2hCUEd2MWNBZ0FBRUFEellEYWVKTTU1eStPbEZWZzFURjNJODdvdm9pdGh0RjYwVmllMGhFOW8zVDJOQ

## 8. Test MLflow logging against the SageMaker MLflow App

This uses the SageMaker MLflow App ARN as the MLflow tracking URI. No Databricks host or token is required.


In [10]:
import mlflow
import tempfile
from pathlib import Path

print("MLflow version:", mlflow.__version__)

mlflow.set_tracking_uri(mlflow_app_arn)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f"{TEAM_ID}_{STUDENT_ID}_mlflow_app_test"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.set_tags({
        "course": COURSE,
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "tracking_backend": "sagemaker_mlflow_app",
        "purpose": "setup_validation",
        "project_name": PROJECT_NAME,
    })

    mlflow.log_params({
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "region": REGION,
        "artifact_store_uri": ARTIFACT_STORE_URI,
        "project_name": PROJECT_NAME,
    })

    mlflow.log_metrics({
        "test_accuracy": 0.888,
        "test_f1": 0.876,
        "test_auc_roc": 0.901,
    })

    summary = {
        "message": "SageMaker MLflow App logging test succeeded.",
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "project_name": PROJECT_NAME,
        "mlflow_app_arn": mlflow_app_arn,
        "experiment_name": EXPERIMENT_NAME,
        "run_id": run.info.run_id,
    }

    with tempfile.TemporaryDirectory() as tmpdir:
        artifact_path = Path(tmpdir) / "mlflow_app_setup_summary.json"
        artifact_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
        mlflow.log_artifact(str(artifact_path), artifact_path="setup_test")

    run_id = run.info.run_id

print("Logged test run successfully.")
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", run_name)
print("Run ID:", run_id)


MLflow version: 3.15.1


🏃 View run team05_s502_mlflow_app_test at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/28568275ee4d47f7a3e6eaf8e98f0139
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
Logged test run successfully.
Experiment: ITI113/team05/Experiment1
Run name: team05_s502_mlflow_app_test
Run ID: 28568275ee4d47f7a3e6eaf8e98f0139


## 9. Search the test run

This confirms that the run is visible through the MLflow API.


In [11]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    raise RuntimeError(f"Experiment not found: {EXPERIMENT_NAME}")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.student_id = '{STUDENT_ID}' and tags.team_id = '{TEAM_ID}'",
    order_by=["attributes.start_time DESC"],
    max_results=10,
)

runs[["run_id", "tags.mlflow.runName", "metrics.test_accuracy", "metrics.test_f1", "metrics.test_auc_roc"]]


,run_id,tags.mlflow.runName,metrics.test_accuracy,metrics.test_f1,metrics.test_auc_roc
0,28568275ee4d47f7a3e6eaf8e98f0139,team05_s502_mlflow_app_test,0.888,0.876,0.901
1,fbdd5ccb9e8a4c4b96bc201b75b0f150,team05_s502_stage10_endpoint_deployment,NaN,NaN,NaN
2,c4a6a3bf9ee547b2b72de0fb9939e056,team05_s502_stage10_endpoint_deployment,NaN,NaN,NaN
3,52280b3f02564a69bae09d5d36f177ae,team05_s502_stage9_final_governance,NaN,NaN,NaN
4,ec12625db7bb406180747a5eca9dac9a,team05_s502_stage8_locked_holdout_final,NaN,NaN,NaN
5,baf87de5eb064d3394aaec1af969caa8,team05_s502_stage7_responsible_ai,NaN,NaN,NaN
6,af027dfa2a4945a7a2450306e00df830,team05_s502_random_forest_full_stage6_tuning,NaN,NaN,NaN
7,49586239a18349d08eb84573721f4eae,team05_s502_xgboost_full_stage6_tuning,NaN,NaN,NaN
8,3fa886a7a47748e9ad76945eef2cf04f,team05_s502_logistic_regression_full_stage6_tu...,NaN,NaN,NaN
9,f26f52da349a4cb9ba41e5e1a1e713e1,team05_s502_random_forest_full_stage6_tuning,NaN,NaN,NaN


## 10. Output values to copy into future notebooks

Use these values in the alternative baseline MLflow notebook and post-pipeline logging notebook.


In [12]:
print("# Copy these into future notebooks")
print(f'REGION = "{REGION}"')
print(f'MLFLOW_APP_ARN = "{mlflow_app_arn}"')
print(f'EXPERIMENT_NAME = "{EXPERIMENT_NAME}"')
print(f'TEAM_ID = "{TEAM_ID}"')
print(f'STUDENT_ID = "{STUDENT_ID}"')
print(f'PROJECT_NAME = "{PROJECT_NAME}"')

# Save locally for reuse by later project notebooks.
config = {
    "REGION": REGION,
    "MLFLOW_APP_ARN": mlflow_app_arn,
    "EXPERIMENT_NAME": EXPERIMENT_NAME,
    "TEAM_ID": TEAM_ID,
    "STUDENT_ID": STUDENT_ID,
    "PROJECT_NAME": PROJECT_NAME,
    "ARTIFACT_STORE_URI": ARTIFACT_STORE_URI,
    "MLFLOW_APP_NAME": MLFLOW_APP_NAME,
    "MLFLOW_APP_TAGS": MLFLOW_APP_TAGS,
}

savepath = Path(
    f"mlflow_app_config_{TEAM_ID.lower()}_{STUDENT_ID.lower()}.json"
)

savepath.write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)

print(f"\nSaved local config: {savepath.name}")
print(f"Full path: {savepath.resolve()}")


# Copy these into future notebooks
REGION = "ap-southeast-1"
MLFLOW_APP_ARN = "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-ZOP45XZVBR5D"
EXPERIMENT_NAME = "ITI113/team05/Experiment1"
TEAM_ID = "team05"
STUDENT_ID = "s502"
PROJECT_NAME = "Heart_Attack_Risk_Assessment"

Saved local config: mlflow_app_config_team05_s502.json
Full path: /home/sagemaker-user/Heart_Attack_Risk_Assessment/mlflow_app_config_team05_s502.json


## 11. Minimal code for later notebooks

After this setup works, future project notebooks can load the saved JSON configuration instead of copying the ARN manually:

```python
import json
import mlflow
from pathlib import Path

CONFIG_FILE = Path("mlflow_app_config_team05_s502.json")

with open(CONFIG_FILE, "r") as f:
    config = json.load(f)

REGION = config["REGION"]
MLFLOW_APP_ARN = config["MLFLOW_APP_ARN"]
EXPERIMENT_NAME = config["EXPERIMENT_NAME"]
TEAM_ID = config["TEAM_ID"]
STUDENT_ID = config["STUDENT_ID"]
PROJECT_NAME = config["PROJECT_NAME"]

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(
    run_name=f"{TEAM_ID}_{STUDENT_ID}_example"
):
    mlflow.set_tags({
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "project_name": PROJECT_NAME,
    })
```

For this project the expected local config file is:

`mlflow_app_config_team05_s502.json`

Keep this file in the `Heart_Attack_Risk_Assessment` SageMaker Studio project folder so later notebooks can reuse the same Team05 MLflow App.


## 12. Cleanup note

For a class, normally keep the MLflow App and let students reuse it. If you need to remove it later, delete the MLflow App from SageMaker and then delete the artifact prefix in S3 only after confirming no runs are needed.

Do **not** delete the artifact store while students still need their experiment artifacts.
